# M12.2 — Marginal weighting diagnostics (CPU only)

## A. Motivation and hypothesis
**Preparation, not training.** Compare frozen M12.1 global Mahalanobis sample weights with proposed independent weights for `[chirp_mass, total_mass, chi_eff]`. The supplied scientific context reports better M12.1 slopes and upper-mass bias, but inconsistent lower-mass and extreme-global-distance gains; this notebook does not re-evaluate or establish those results.

Hypothesis: global weights may increase all three task losses when only one target is peripheral. Marginal weights may instead redistribute optimization emphasis toward the peripheral target on that same row. Meaningfully different assignments and residual-dependent objective changes would support considering an experiment, not establish better generalization or multi-task interference.

Freeze final G1, all splits, preprocessing, saved label scaling, architecture, optimizer and evaluation definitions. Alpha stays 0.5. Only TRAIN labels enter reference fitting and diagnostics. No validation/calibration/test labels, predictions or real events are read. No inference, training, bootstrap, GPU access, or outputs are written. All cells are delivered unexecuted.

Scope note: AGENTS.md historically reserved M12.2 for balanced sampling. The explicit current request uses this name for **marginal-weighting diagnostics**; no balanced sampling or training is implemented. Reconcile stage naming before a future experiment, without retroactively relabeling historical results.

References: [M12.0](M12_0_frozen_baseline_and_acceptance.ipynb), [M12.1 preparation](M12_1_weighted_mse.ipynb), and the marginal-peripherality definitions in [validation evaluation](M12_1_validation_evaluation.ipynb). No historical metric values are used to choose thresholds here.

## B. Frozen train data and provenance
Use the same `src.paths` convention as M12.0: `CBC_PE_DATA_ROOT` overrides the frozen M11 config root. Read only split metadata, saved train-only label statistics, and selected physical HDF5 train rows in sorted original-row order. Use the existing M12.1 bounded sorted-row read pattern, not full strain or full-label reads. `train_ids[i]` identifies every weight row; sorted positions are not replacement source IDs.

The saved scaler remains unchanged. Geometry uses physical train mean and population standard deviation (ddof=0), separately from the float32 saved training scaler. Hash the small split/stat files against the frozen M12.0 contract. File size/mtime alone is not dataset-content proof; train IDs, label hashes and geometry checks strengthen provenance. Missing artifacts or mismatches stop the analysis.

In [1]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import h5py
from IPython.display import display

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "src/paths.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open from cbc_pe")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.paths import resolve_data_root, resolve_processed_artifact
from src.models.sample_weighting import mahalanobis_sample_weights, array_hash

LABELS = ["chirp_mass", "total_mass", "chi_eff"]
ALPHA = 0.5
MIN_COUNT = 100
CONFIG_PATH = ROOT / "configs/experiments/train_m11_final_G1_500k.json"
config = json.loads(CONFIG_PATH.read_text())
DATA_ROOT = resolve_data_root(config_data_root=config["data_root"])
DATASET_ID = "M11_final_G1_500k"
PATHS = {k:resolve_processed_artifact(data_root=DATA_ROOT, dataset_id=DATASET_ID,
    file_name=config["dataset"][k], allow_legacy_flat=False)
    for k in ("dataset_file", "split_file", "label_stats_file")}
CONTRACT_PATH = ROOT / "notebooks/m12/outputs/m12_0_baseline_contract.json"

def require(ok, message):
    if not ok:
        raise ValueError(message)

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

require(config["dataset"]["dataset_id"] == DATASET_ID, "Wrong dataset config")
require(all(p.is_file() for p in PATHS.values()) and CONTRACT_PATH.is_file(), "Missing frozen artifacts")
frozen = json.loads(CONTRACT_PATH.read_text())
require(frozen["dataset_id"] == DATASET_ID and frozen["labels"] == LABELS
        and frozen["schema_version"] == 1, "Wrong frozen contract")
geometry = frozen["geometry"]
for key in ("split_file", "label_stats_file"):
    require(sha256_file(PATHS[key]) == frozen["provenance"]["local_artifacts"][key]["sha256"],
            f"Changed frozen artifact: {key}")
del frozen  # No historical test metrics or real-event results are used.
with np.load(PATHS["split_file"], allow_pickle=False) as z:
    train_ids = z["train_idx"].copy()
    require(int(z["seed"].item()) == 123, "Split seed changed")
require(train_ids.shape == (400000,) and np.issubdtype(train_ids.dtype, np.integer)
        and len(np.unique(train_ids)) == len(train_ids), "Invalid train IDs")
train_ids = np.sort(train_ids.astype(np.int64))
require(((train_ids >= 0) & (train_ids < 500000)).all(), "Out-of-range train IDs")
require(array_hash(train_ids) == geometry["train_indices_sha256"], "Train identity mismatch")
with np.load(PATHS["label_stats_file"], allow_pickle=False) as z:
    require(z["label_names"].tolist() == LABELS, "Scaler label order")
    require(np.array_equal(np.sort(z["train_idx"]), train_ids), "Scaler train membership")
    saved_mean, saved_std = z["y_mean"].copy(), z["y_std"].copy()
require(saved_mean.shape == saved_std.shape == (3,) and np.isfinite(saved_mean).all()
        and np.isfinite(saved_std).all() and (saved_std > 0).all(), "Invalid frozen scaler")
with h5py.File(PATHS["dataset_file"], "r") as h:
    require(h.attrs.get("experiment_id") == DATASET_ID
            and h.attrs.get("labels") == ",".join(LABELS), "HDF5 identity/label order")
    require(h["y"].shape == (500000,3), "Wrong physical label shape")
    membership = h["split"].asstr()[:]
    require(np.array_equal(np.flatnonzero(membership == "train"),train_ids), "HDF5 train mismatch")
    train_y = np.empty((len(train_ids),3),dtype=np.float64)
    for start in range(0,len(train_ids),4096):
        train_y[start:start+4096] = h["y"][train_ids[start:start+4096]]
del membership
require(np.isfinite(train_y).all(), "Nonfinite train labels")
stat = PATHS["dataset_file"].stat()
provenance = dict(dataset_id=DATASET_ID, split="train", labels=LABELS,
    paths={k:str(p) for k,p in PATHS.items()},
    config_sha256=sha256_file(CONFIG_PATH), contract_sha256=sha256_file(CONTRACT_PATH),
    weighting_source_sha256=sha256_file(ROOT / "src/models/sample_weighting.py"),
    split_sha256=sha256_file(PATHS["split_file"]), stats_sha256=sha256_file(PATHS["label_stats_file"]),
    train_ids_sha256=array_hash(train_ids), physical_train_labels_sha256=array_hash(train_y),
    dataset_size=stat.st_size, dataset_mtime_ns=stat.st_mtime_ns,
    versions=dict(python=sys.version,numpy=np.__version__,pandas=pd.__version__,h5py=h5py.__version__))
print("Loaded physical TRAIN labels only:", train_y.shape)

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/afs/ciemat.es/user/v/vserrano/miniconda3/envs/gw-env/lib/python3.10/site-packages/traitlets/traitlets.py", line 632, in get
    value = obj._trait_values[self.name]
KeyError: '_control_lock'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/afs/ciemat.es/user/v/vserrano/miniconda3/envs/gw-env/lib/python3.10/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/afs/ciemat.es/user/v/vserrano/miniconda3/envs/gw-env/lib/python3.10/site-packages/ipykernel/kernelbase.py", line 347, in dispatch_control
    async with self._control_lock:
  File "/afs/ciemat.es/user/v/vserrano/miniconda3/envs/gw-env/lib/python3.10/site-packages/traitlets/traitlets.py", line 687, in __get__
    return t.cast(G, self.get(obj, cls))  # the G should encode the Optional
  File "/afs/ciemat.es/user/v/vserrano/min

Loaded physical TRAIN labels only: (400000, 3)


## C. Reproduce M12.1 global reference
Call the unchanged `mahalanobis_sample_weights` implementation once on sorted physical train labels: mean/std (ddof=0), covariance (ddof=1), standardized covariance inverse or pseudoinverse with rcond=1e-12, and right-side train empirical CDF. Global g_i = (1 + 0.5 q_i) / mean_train(1 + 0.5 q).

Check moments, precision, rank and sorted-distance hash against M12.0; no evaluation-derived mapping is substituted. Record the helper's normalized-weight hash as a **new reproduction hash**, not a historical checkpoint verification. An optional already-recorded M12.1 normalized-weight hash may be pasted below for comparison. No checkpoint, validation archive or prediction search is needed. Numerical-library differences causing a hash mismatch require investigation, not an automatic tolerance relaxation.

In [2]:
RECORDED_M12_1_WEIGHT_SHA256 = None  # Optional known normalized_weight_sha256, never invent it.
global_w, global_q, global_meta = mahalanobis_sample_weights(train_y, alpha=ALPHA)
for key in ("n_train", "rank", "pinv_rcond", "reference_distance_sha256"):
    require(global_meta[key] == geometry[key], f"Frozen geometry mismatch: {key}")
for key in ("mean", "std_ddof0", "covariance_ddof1", "scaled_covariance", "precision"):
    np.testing.assert_allclose(global_meta[key], geometry[key], rtol=1e-12, atol=1e-12)
require(array_hash(global_w) == global_meta["normalized_weight_sha256"], "Weight hash mismatch")
if RECORDED_M12_1_WEIGHT_SHA256 is not None:
    require(array_hash(global_w) == RECORDED_M12_1_WEIGHT_SHA256, "Recorded M12.1 weight mismatch")
else:
    print("Historical M12.1 weight hash not supplied; implementation and M12.0 reference agreement checked.")
mu = np.asarray(global_meta["mean"],dtype=np.float64)
sigma = np.asarray(global_meta["std_ddof0"],dtype=np.float64)
provenance["global_weighting"] = global_meta
provenance["recorded_M12_1_weight_hash"] = RECORDED_M12_1_WEIGHT_SHA256
display(pd.DataFrame(dict(label=LABELS, geometry_mu=mu, geometry_sigma=sigma,
                          frozen_label_mean=saved_mean, frozen_label_std=saved_std)))

Historical M12.1 weight hash not supplied; implementation and M12.0 reference agreement checked.


,label,geometry_mu,geometry_sigma,frozen_label_mean,frozen_label_std
0,chirp_mass,37.463257,16.508970,37.463547,16.508060
1,total_mass,94.986031,34.705112,94.985977,34.703293
2,chi_eff,-0.000219,0.440367,-0.000219,0.440338


## D–E. Marginal weights and mathematical checks
For each label k: d_ik = |y_ik − μ_train,k| / σ_train,k; q_ik = count(d_train,k ≤ d_ik)/N_train. Include the row itself and all ties (`side="right"`), exactly like M12.1. Independently normalize each column of 1 + 0.5q to train mean one. Distances/ranks/normalization use float64; weights are stored **in memory only** as float32, shape (400000,3). No 400k×3 artifact is exported.

For standardized residuals r, L_global = mean_ik(g_i r_ik²) and L_marginal = mean_ik(w_ik r_ik²). This preserves equal nominal target contributions and uses train-wide normalization, not minibatch weight sums. Each task's empirical weighting may still alter gradients on shared network parameters. Alpha=0 gives unit weights and standard MSE; global weights broadcast across labels recover the original sample-weighted objective. Float32 reductions need tolerance, not a claim of bitwise-identical training.

In [3]:
marginal_q = np.empty_like(train_y)
marginal_cdf_hashes = {}
for j,label in enumerate(LABELS):
    d = np.abs((train_y[:,j]-mu[j])/sigma[j])
    reference = np.sort(d)
    marginal_q[:,j] = np.searchsorted(reference,d,side="right") / len(train_ids)
    marginal_cdf_hashes[label] = array_hash(reference)
del d, reference

def marginal_weights(q, alpha):
    require(np.isfinite(alpha) and alpha >= 0, "Invalid alpha")
    raw = 1 + alpha*q
    return (raw/raw.mean(axis=0)).astype(np.float32)

marginal_w = marginal_weights(marginal_q,ALPHA)
require(marginal_w.shape == (len(train_ids),3) and global_w.shape == (len(train_ids),), "Weight shape")
for q in (global_q,marginal_q):
    require(np.isfinite(q).all() and ((q >= 0) & (q <= 1)).all(), "CDF bounds")
for weights in (global_w,marginal_w):
    require(np.isfinite(weights).all() and (weights > 0).all(), "Weights must be finite and positive")
    np.testing.assert_allclose(weights.mean(axis=0,dtype=np.float64),1.,rtol=1e-7,atol=1e-7)
# Tiny deterministic arrays: no model or training run.
toy_q = np.array([[0.2,0.8,1.0],[0.6,0.3,0.7],[1.0,0.5,0.2]])
r = np.array([[1.,-2.,3.],[-1.,.5,2.],[0.,2.,-1.]])
unit = marginal_weights(toy_q,0.)
require(np.array_equal(unit,np.ones_like(unit)), "alpha=0 must give ones")
np.testing.assert_allclose(np.mean(unit*r*r),np.mean(r*r),rtol=1e-12)
g_toy = 1 + ALPHA*toy_q[:,0]; g_toy /= g_toy.mean()
np.testing.assert_allclose(np.mean(g_toy[:,None]*r*r),np.mean(g_toy*np.mean(r*r,axis=1)),rtol=1e-12)
# Gradient w.r.t. residual/prediction is 2*w*r/(N*K); alpha=0 is standard MSE.
np.testing.assert_array_equal(2*unit*r/r.size,2*r/r.size)
# Right-side tie check, including self.
np.testing.assert_allclose(np.searchsorted([0.,1.,1.,2.],[0.,1.,1.,2.],side="right")/4,[.25,.75,.75,1.])
provenance["marginal"] = dict(alpha=ALPHA,normalization="per_label_train_mean_one",
    weight_sha256=array_hash(marginal_w),cdf_sha256=marginal_cdf_hashes,
    row_ids_sha256=array_hash(train_ids),label_order=LABELS)
print("Finite, positive, mean-one, shape, ties and alpha=0 checks passed.")

Finite, positive, mean-one, shape, ties and alpha=0 checks passed.


## F. Assignment differences, not just histograms
Pearson correlations compare aligned global/marginal weights and target-to-target weights; Spearman correlations use average ranks for ties. Mean absolute differences compare the same row/task pair. Quantiles summarize marginal distributions and assignment differences separately. Per-row max−min of the three marginal columns measures task disagreement; it is identically zero for a broadcast global weight. No correlation cutoff is chosen after observing the data. Tables suffice here; no figures are needed.

In [4]:
comparison = pd.DataFrame({"global":global_w, **{label:marginal_w[:,j] for j,label in enumerate(LABELS)}})
display(comparison.corr(method="pearson"))
# pandas average ranks + Pearson avoids introducing a SciPy dependency for Spearman.
display(comparison.rank(method="average").corr(method="pearson"))
quantiles = [0.,.1,.5,.9,.99,1.]
display(comparison.quantile(quantiles).rename_axis("quantile"))
assignment_rows = []
for j,label in enumerate(LABELS):
    delta = marginal_w[:,j].astype(np.float64)-global_w
    assignment_rows.append(dict(label=label,mean_abs_difference=np.abs(delta).mean(),
        signed_mean_difference=delta.mean(),fraction_above_global=np.mean(delta > 0),
        fraction_below_global=np.mean(delta < 0),median_abs_difference=np.quantile(abs(delta),.5),
        p90_abs_difference=np.quantile(abs(delta),.9),max_abs_difference=abs(delta).max()))
assignment_summary = pd.DataFrame(assignment_rows)
display(assignment_summary)
row_disagreement = np.ptp(marginal_w,axis=1)
display(pd.DataFrame({"quantile":quantiles,"marginal_row_max_minus_min":np.quantile(row_disagreement,quantiles),
                      "global_row_max_minus_min":0.}))
del comparison, delta

,global,chirp_mass,total_mass,chi_eff
global,1.000000,0.562747,0.386412,0.555215
chirp_mass,0.562747,1.000000,0.856426,-0.004719
total_mass,0.386412,0.856426,1.000000,-0.045059
chi_eff,0.555215,-0.004719,-0.045059,1.000000


,global,chirp_mass,total_mass,chi_eff
global,1.000000,0.562747,0.386412,0.555215
chirp_mass,0.562747,1.000000,0.856426,-0.004719
total_mass,0.386412,0.856426,1.000000,-0.045059
chi_eff,0.555215,-0.004719,-0.045059,1.000000


,global,chirp_mass,total_mass,chi_eff
quantile,,,,
0.00,0.800001,0.800001,0.800001,0.800001
0.10,0.840000,0.840000,0.840000,0.840000
0.50,1.000000,1.000000,1.000000,1.000000
0.90,1.159999,1.159999,1.159999,1.159999
0.99,1.195999,1.195999,1.195999,1.195999
1.00,1.199999,1.199999,1.199999,1.199999


,label,mean_abs_difference,signed_mean_difference,fraction_above_global,fraction_below_global,median_abs_difference,p90_abs_difference,max_abs_difference
0,chirp_mass,0.086321,-2.235174e-12,0.562180,0.437818,0.075667,0.167224,0.377509
1,total_mass,0.099042,-1.192093e-11,0.622550,0.377447,0.082096,0.198896,0.399846
2,chi_eff,0.085867,-4.470348e-12,0.578727,0.421272,0.073969,0.167596,0.398252


,quantile,marginal_row_max_minus_min,global_row_max_minus_min
0,0.00,0.000358,0.0
1,0.10,0.044128,0.0
2,0.50,0.143791,0.0
3,0.90,0.294328,0.0
4,0.99,0.370171,0.0
5,1.00,0.399573,0.0


## G. Discordant mass/spin regions
Define peripheral as train marginal q ≥ 0.80, central as q ≤ 0.50. These are fixed diagnostic thresholds; the middle band is excluded from the discordant contrasts, not from global diagnostics. For each mass separately compare (mass peripheral AND spin central) versus (mass central AND spin peripheral). Counts and fractions are explicit; mass regions may overlap and are not added together. Report all three target weights within each region. A positive delta means more emphasis than global weighting, not a predicted error improvement. Suppress summaries for n<100.

In [5]:
regions = {}
for j,label in enumerate(LABELS[:2]):
    regions[label+"_peripheral_spin_central"] = (marginal_q[:,j] >= .8) & (marginal_q[:,2] <= .5)
    regions[label+"_central_spin_peripheral"] = (marginal_q[:,j] <= .5) & (marginal_q[:,2] >= .8)

def region_summary(name, mask):
    n = int(mask.sum()); supported = n >= MIN_COUNT
    rows = []
    for j,label in enumerate(LABELS):
        g = float(global_w[mask].mean(dtype=np.float64)) if supported else np.nan
        w = float(marginal_w[mask,j].mean(dtype=np.float64)) if supported else np.nan
        rows.append(dict(region=name,label=label,n=n,fraction=n/len(train_ids),supported=supported,
            global_mean=g,marginal_mean=w,delta_mean=w-g,
            fraction_increased=float(np.mean(marginal_w[mask,j] > global_w[mask])) if supported else np.nan))
    return rows

regional = pd.DataFrame([row for name,mask in regions.items() for row in region_summary(name,mask)])
display(regional)

,region,label,n,fraction,supported,global_mean,marginal_mean,delta_mean,fraction_increased
0,chirp_mass_peripheral_spin_central,chirp_mass,41483,0.103707,True,1.070056,1.161260,0.091205,0.978232
1,chirp_mass_peripheral_spin_central,total_mass,41483,0.103707,True,1.070056,1.153203,0.083148,0.970856
2,chirp_mass_peripheral_spin_central,chi_eff,41483,0.103707,True,1.070056,0.898612,-0.171444,0.000000
3,chirp_mass_central_spin_peripheral,chirp_mass,38793,0.096982,True,1.081206,0.901320,-0.179886,0.000000
4,chirp_mass_central_spin_peripheral,total_mass,38793,0.096982,True,1.081206,0.911922,-0.169284,0.002732
5,chirp_mass_central_spin_peripheral,chi_eff,38793,0.096982,True,1.081206,1.159429,0.078222,0.927538
6,total_mass_peripheral_spin_central,chirp_mass,42221,0.105552,True,1.075100,1.154034,0.078934,0.977594
7,total_mass_peripheral_spin_central,total_mass,42221,0.105552,True,1.075100,1.160460,0.085361,1.000000
8,total_mass_peripheral_spin_central,chi_eff,42221,0.105552,True,1.075100,0.898600,-0.176500,0.000000
9,total_mass_central_spin_peripheral,chirp_mass,42343,0.105857,True,1.097534,0.924860,-0.172674,0.000000


## H. Compact joint-peripherality summary
Eight exhaustive three-bit states indicate whether chirp_mass, total_mass, chi_eff each have q≥0.80. Here False means **nonperipheral (<0.80)**, not central (≤0.50). Each row belongs to exactly one state, ties at 0.80 go to peripheral. Show support and target-specific emphasis changes. Unequal counts reflect correlated labels; these associations are not a causal separation of tasks.

In [6]:
flags = marginal_q >= .8
joint_code = flags[:,0].astype(int)*4 + flags[:,1].astype(int)*2 + flags[:,2].astype(int)
joint_rows = []
for state in range(8):
    name = format(state,"03b")
    joint_rows.extend(region_summary(name,joint_code == state))
joint = pd.DataFrame(joint_rows)
require((joint.groupby("label").n.sum().reindex(LABELS) == len(train_ids)).all(), "Incomplete joint partition")
display(joint[["region","label","n","supported","global_mean","marginal_mean","delta_mean"]])
del flags, joint_code

,region,label,n,supported,global_mean,marginal_mean,delta_mean
0,000,chirp_mass,244210,True,0.937205,0.952582,0.015377
1,000,total_mass,244210,True,0.937205,0.955950,0.018745
2,000,chi_eff,244210,True,0.937205,0.961096,0.023891
3,001,chirp_mass,62361,True,1.099394,0.960628,-0.138766
4,001,total_mass,62361,True,1.099394,0.949379,-0.150014
5,001,chi_eff,62361,True,1.099394,1.159995,0.060601
6,010,chirp_mass,11050,True,1.071840,1.091822,0.019982
7,010,total_mass,11050,True,1.071840,1.138877,0.067037
8,010,chi_eff,11050,True,1.071840,0.957463,-0.114377
9,011,chirp_mass,2378,True,1.153972,1.092712,-0.061260


## I. Fixed synthetic residual objectives — no predictor claim
Use deterministic, dimensionless **standardized-target residuals**, not physical mass/spin errors. Compare unweighted, global and marginal MSE overall and per label. All three task contributions enter the overall objective with factor 1/3.

Scenarios are fixed before execution: constant residuals (normalization sanity), row-ID sinusoidal residuals (label-agnostic control), own-label percentile-dependent residuals, and each discordant region with extra residual only on its named mass or spin target. The latter are deliberately aligned stress cases, not observed model failures. No real TRAIN prediction artifact was established in this targeted preparation; none is loaded and no inference is launched. A future compatible artifact would need explicit row/scaler/checkpoint checks before use, outside this notebook's current scope.

For squared residual s, L_marginal−L_global = mean((w−g)s). Since each weight column has mean one, near-constant errors can give equal scalar losses even with different gradient coefficients. Conversely, a scalar difference under a hand-designed residual proves objective sensitivity, not expected model improvement. The analytical residual-gradient difference is 2(w−g)r/(3N); network-gradient directions also depend on the model Jacobian and are not computed here.

In [7]:
objective_rows = []

def compare_objectives(name, residual):
    require(residual.shape == (len(train_ids),3) and np.isfinite(residual).all(), "Residual shape/finite")
    squared = residual*residual
    base = squared.mean(axis=0)
    glob = (global_w[:,None]*squared).mean(axis=0)
    marg = (marginal_w*squared).mean(axis=0)
    identity = ((marginal_w-global_w[:,None])*squared).mean(axis=0)
    np.testing.assert_allclose(marg-glob,identity,rtol=1e-6,atol=1e-7)
    gradient_change = np.mean(np.abs((marginal_w-global_w[:,None])*residual),axis=0)
    for j,label in enumerate(LABELS):
        objective_rows.append(dict(scenario=name,label=label,unweighted_MSE=base[j],
            global_MSE=glob[j],marginal_MSE=marg[j],delta=marg[j]-glob[j],
            mean_abs_residual_gradient_delta=2*gradient_change[j]/residual.size))
    objective_rows.append(dict(scenario=name,label="equal_target_mean",unweighted_MSE=base.mean(),
        global_MSE=glob.mean(),marginal_MSE=marg.mean(),delta=(marg-glob).mean(),
        mean_abs_residual_gradient_delta=2*gradient_change.mean()/residual.size))

N = len(train_ids)
compare_objectives("constant_unit",np.ones((N,3),dtype=np.float64))
# Stable original-row identity, no random-number generator or fitted residual model.
control = np.sin(train_ids[:,None]*np.array([.017,.023,.031])[None,:])
compare_objectives("row_id_sine_control",control)
del control
compare_objectives("own_marginal_peripherality",.25+marginal_q)
for j,label in enumerate(LABELS[:2]):
    for suffix,target in (("_peripheral_spin_central",j),("_central_spin_peripheral",2)):
        name = label+suffix
        if regions[name].sum() < MIN_COUNT:
            print("Unsupported synthetic regional stress case skipped:",name)
            continue
        residual = np.ones((N,3),dtype=np.float64)
        residual[regions[name],target] = 2.
        compare_objectives(name,residual)
        del residual
objectives = pd.DataFrame(objective_rows)
display(objectives)
constant = objectives[objectives.scenario.eq("constant_unit")]
np.testing.assert_allclose(constant[["unweighted_MSE","global_MSE","marginal_MSE"]],1.,rtol=1e-7,atol=1e-7)

,scenario,label,unweighted_MSE,global_MSE,marginal_MSE,delta,mean_abs_residual_gradient_delta
0,constant_unit,chirp_mass,1.000000,1.000000,1.000000,-2.235101e-12,1.438678e-07
1,constant_unit,total_mass,1.000000,1.000000,1.000000,-1.192091e-11,1.650708e-07
2,constant_unit,chi_eff,1.000000,1.000000,1.000000,-4.470313e-12,1.431122e-07
3,constant_unit,equal_target_mean,1.000000,1.000000,1.000000,-6.208774e-12,1.506836e-07
4,row_id_sine_control,chirp_mass,0.500000,0.500070,0.500003,-6.727791e-05,9.162122e-08
5,row_id_sine_control,total_mass,0.499991,0.499994,0.499958,-3.686699e-05,1.052241e-07
6,row_id_sine_control,chi_eff,0.499995,0.500031,0.499988,-4.259154e-05,9.110578e-08
7,row_id_sine_control,equal_target_mean,0.499995,0.500032,0.499983,-4.891215e-05,9.598370e-08
8,own_marginal_peripherality,chirp_mass,0.645835,0.674902,0.695835,2.093349e-02,1.031190e-07
9,own_marginal_peripherality,total_mass,0.645835,0.667783,0.695835,2.805268e-02,1.107005e-07


## J. Interpretation boundaries
Similar marginal weight distributions are expected because independent empirical ranks are near-uniform when ties are rare. They do **not** imply that the same sample receives the same task weight. Judge aligned correlations, absolute differences, discordant-region shifts and per-row task spread together. Ties can alter the distribution; right-side CDF ranks and per-column normalization are retained exactly.

A different coefficient tensor is a different objective even when one synthetic scalar loss coincides. Its relevance to GPU training depends on residual/gradient structure not measured here. Correlated mass labels can retain similar assignments; spin may differ. Such associations do not establish multi-task interference or show that shared-gradient conflict is reduced. Absolute marginal distances fold low/high tails together; symmetry of weights does not guarantee symmetric bias improvement.

The full train reference includes all 400000 rows. Historical block batching/drop_last remains frozen and omits its last partial batch; a future experiment must retain that behavior and must not renormalize within batches. This diagnostic reports full-train objectives, not the realized minibatch optimization trajectory. Frozen label scalers and unweighted validation checkpoint selection must stay unchanged.

## K. Decision record and unresolved assumptions
No automatic go/no-go cutoff is inferred from these diagnostics. A GPU experiment would require an explicit hypothesis, acceptance criteria and separate authorization. This notebook changes no trainer/config and schedules no work.

| Question | Evidence to inspect | Interpretation / next decision |
|---|---|---|
| Is the global reference reproduced? | Train IDs, frozen moments, distance hash; optional recorded weight hash | Stop on mismatch; resolve provenance first |
| Are proposed weights mathematically valid? | Finite positive N×3, mean-one columns, alpha=0 checks | Necessary implementation checks, not performance evidence |
| Are assignments distinct? | Aligned correlations, MA differences, row max−min, discordant regions | Decide prospectively what magnitude is sufficient; no arbitrary cutoff here |
| Which targets gain emphasis? | Joint-state and conditional target means | Establish whether the proposal addresses the stated hypothesis |
| Does the objective respond differently? | Fixed residual scenarios and gradient-coefficient differences | Controlled sensitivity only; not model error estimates |
| Is a GPU trial scientifically justified? | Supported, nontrivial assignment differences plus a preregistered validation plan | Human scientific decision; no overall score or automatic launch |

Unresolved: minimum meaningful assignment/objective difference; actual TRAIN predictor residual/gradient behavior; source dependence beyond original row IDs; historical checkpoint-byte/weight-hash verification if not supplied; stage naming versus the older balanced-sampling plan; numerical validation guardrails and mass/spin tradeoffs. Existing supplied M12.1 qualitative results are motivation, not recomputed evidence.

**Manual execution:** use the existing local Python 3.10 environment with project dependencies and the normal `CBC_PE_DATA_ROOT`/M11-config root. Run B–E sequentially and stop on a contract failure. Then run F–I and review J–K. Do not Run All. The helper imports PyTorch but no tensor/model/GPU operations are invoked here. Missing frozen inputs remain blockers; do not generate replacements. No packages, signals, validation/calibration/test labels or outputs are needed. No large weight artifact is written.

## L. Scientific conclusions and decision — M12.2 marginal weighting

### L.1. Objective and methodology

This diagnostic investigated whether parameter-specific marginal weighting provides a meaningfully different optimization objective from the global Mahalanobis weighting implemented in M12.1.

Both approaches use the same frozen G1 training population, a weighting strength of $\alpha=0.5$, and weights normalized to unit mean. The difference lies in how sample importance is assigned to the three regression targets.

M12.1 assigns one global Mahalanobis-based weight to all three errors of a given sample. M12.2 instead computes three independent weights from the train-only marginal peripherality of chirp mass, total mass, and effective spin.

No model training, inference, or validation-based hyperparameter optimization was performed in this diagnostic.

### L.2. Main findings

**Equivalent marginal weight distributions, different assignments.** All four weight distributions are approximately bounded between 0.8 and 1.2, as expected from empirical percentile weighting with $\alpha=0.5$ and mean-one normalization. Their similar distributions do not imply equivalent objectives: different samples and different regression targets receive the largest weights.

**Mass and spin peripherality exhibit distinct structures.** The correlation between chirp-mass and total-mass marginal weights is approximately 0.856. In contrast, effective-spin marginal weights are nearly uncorrelated with those of chirp mass (−0.005) and total mass (−0.045). Global Mahalanobis weighting therefore combines partly distinct notions of parameter peripherality.

**Material differences in sample-to-target weighting.** The mean absolute difference between global and marginal weights is approximately 0.086 for chirp mass, 0.099 for total mass, and 0.086 for effective spin. The median within-sample spread among the three marginal weights is approximately 0.144, increasing to approximately 0.294 at the 90th percentile.

**Discordant mass–spin regions reveal the main methodological difference.** For samples with peripheral chirp mass and central effective spin, marginal weighting increases the emphasis on mass errors while reducing the weight assigned to spin errors. The opposite occurs for central chirp mass and peripheral effective spin: marginal weighting preferentially emphasizes spin rather than all three outputs simultaneously.

These regions contain tens of thousands of training examples, so the observed differences are not restricted to isolated outliers.

**Synthetic-loss checks confirm distinct objectives.** Both formulations preserve the expected normalization and recover ordinary MSE when $\alpha=0$. Under artificially constructed residuals that increase with each target's own peripherality, marginal weighting places a stronger penalty on those residuals than global weighting. These checks validate the intended mathematical behavior but provide no evidence of improved predictive accuracy.

### L.3. Scientific interpretation

The diagnostics demonstrate that M12.2 introduces a genuine target-specific redistribution of optimization pressure, rather than a simple rescaling of the M12.1 objective.

This provides a plausible mechanism for addressing some of the limitations observed in M12.1, particularly the deterioration of mass predictions in regions where effective spin is peripheral but mass parameters are relatively central.

Nevertheless, parameter-specific weighting does not directly identify which parameter is poorly predicted, does not eliminate shared-backbone gradient interactions, and does not establish the superiority of separate regression heads.

The results are descriptive and mathematical, not evidence of improved generalization.

### L.4. Decision and next experiment

**Decision: proceed with one controlled M12.2 marginal-weighted MSE training experiment.**

The proposed intervention is:

$$
\mathcal{L}_{\mathrm{M12.2}} =
\frac{1}{3B}\sum_{i=1}^{B}\sum_{k=1}^{3}
w_{ik}(\hat y_{ik}^{\mathrm{std}}-y_{ik}^{\mathrm{std}})^2,
$$

where each $w_{ik}$ is obtained from the training empirical percentile of the absolute standardized deviation of label $k$, using $\alpha=0.5$ and independent mean-one normalization for each label.

The experiment will preserve the frozen G1 dataset and splits, network architecture, initialization seed, optimizer, learning-rate schedule, training budget, input normalization, target scaling, validation objective, and checkpoint-selection policy.

M12.2 will be evaluated against both M11 and M12.1 using the same 40,000 validation samples and predefined acceptance criteria, including the provisional 2% global RMSE degradation guardrail.

Particular attention will be paid to lower-mass bias, extreme global-Mahalanobis regions, and discordant mass–spin peripherality, without changing the previously established primary acceptance criteria.

Calibration, test, and real-event datasets remain excluded from model selection.

### L.5. Limitations and future work

A successful marginal-weighting experiment would demonstrate that target-specific loss weighting can improve the optimization trade-off, but would not by itself prove gradient conflict between the three regression tasks.

If residual limitations remain, subsequent research may investigate shared-backbone gradient alignment, parameter-specific regression heads, or partial architectural specialization.

Balanced sampling remains a separate controlled intervention.

**Status: M12.2 CPU diagnostics completed; one marginal-weighted training experiment scientifically justified.**
